# Dimensional Data Modeling (Kimball)

M07 Lesson 15 introduced star schemas. This lesson goes further into Ralph Kimball's **dimensional modelling** method: designing a warehouse around business processes, with conformed dimensions shared across facts, three types of fact tables, and strategies for dimensions that change over time.

**After this lesson you can:**

1. Plan a warehouse with a bus matrix of business processes and conformed dimensions.
2. Distinguish transaction, periodic snapshot and accumulating snapshot facts.
3. Implement slowly changing dimensions (SCD) of type 1, 2 and 3 in SQL.
4. Use degenerate and junk dimensions, and choose the grain deliberately.

**Data:** the Chinook music store (sales) and NYC taxi trips (operations), in DuckDB.

In [1]:
import sys
import tempfile
import time
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
WORK = Path(tempfile.mkdtemp())                   # all warehouses and projects of this lesson live here

import sqlite3

wh = duckdb.connect()
wh.execute(f"ATTACH '{datasets.fetch('chinook.sqlite')}' AS src (TYPE sqlite, READ_ONLY)")
print(wh.sql("SELECT count(*) AS invoice_lines FROM src.InvoiceLine").df())

   invoice_lines
0           2240


## 1. The Four-Step Design Process

1. **Choose the business process** (sales, inventory, support tickets).
2. **Declare the grain**: what exactly is one fact row? (one invoice line)
3. **Identify the dimensions**: who, what, where, when, how (customer, track, date, employee).
4. **Identify the facts**: the numeric measures at that grain (quantity, revenue).

## 2. The Bus Matrix and Conformed Dimensions

A **bus matrix** lists business processes (rows) and dimensions (columns). A dimension used by several processes must be **conformed**: the same keys and attributes everywhere, so that facts from different processes can be compared.

In [2]:
bus = pd.DataFrame({
    "date": ["x", "x", "x", "x"], "customer": ["x", "x", "", "x"], "track/product": ["x", "", "x", ""],
    "employee": ["x", "x", "", "x"], "playlist": ["", "", "x", ""], "channel": ["", "x", "", "x"],
}, index=["sales (invoice lines)", "support tickets", "playlist curation", "marketing campaigns"])
bus

,date,customer,track/product,employee,playlist,channel
sales (invoice lines),x,x,x,x,,
support tickets,x,x,,x,,x
playlist curation,x,,x,,x,
marketing campaigns,x,x,,x,,x


## 3. Three Types of Fact Tables

| Type | One row per | Example | Rows grow with |
| :--- | :--- | :--- | :--- |
| **transaction** | event | an invoice line, a taxi trip | events |
| **periodic snapshot** | entity × period | revenue per customer per month | entities × periods |
| **accumulating snapshot** | process instance, updated as it moves | an order with order/ship/deliver dates | instances (rows are updated) |

In [3]:
wh.execute("""
CREATE TABLE fact_sales AS                                   -- transaction fact: one row per invoice line
SELECT il.InvoiceLineId AS sales_key, il.InvoiceId AS invoice_number,           -- invoice_number: a degenerate dimension
       CAST(strftime(CAST(i.InvoiceDate AS DATE), '%Y%m%d') AS INTEGER) AS date_key,
       i.CustomerId AS customer_key, il.TrackId AS track_key, c.SupportRepId AS employee_key,
       il.Quantity AS quantity, il.UnitPrice * il.Quantity AS revenue
FROM src.InvoiceLine il JOIN src.Invoice i USING (InvoiceId) JOIN src.Customer c USING (CustomerId)
""")
wh.execute("""
CREATE TABLE fact_customer_month AS                          -- periodic snapshot: one row per customer and month
WITH sales AS (SELECT *, date_trunc('month', CAST(strptime(CAST(date_key AS VARCHAR), '%Y%m%d') AS DATE)) AS month FROM fact_sales),
     months AS (SELECT DISTINCT month FROM sales),
     custs AS (SELECT DISTINCT customer_key FROM sales),
     monthly AS (
         SELECT m.month, c.customer_key, coalesce(sum(s.revenue), 0) AS revenue, count(DISTINCT s.invoice_number) AS invoices
         FROM months m CROSS JOIN custs c
         LEFT JOIN sales s ON s.customer_key = c.customer_key AND s.month = m.month
         GROUP BY m.month, c.customer_key)
SELECT *, sum(revenue) OVER (PARTITION BY customer_key ORDER BY month) AS lifetime_revenue FROM monthly
""")
print(wh.sql("SELECT 'fact_sales' AS t, count(*) AS rows FROM fact_sales UNION ALL SELECT 'fact_customer_month', count(*) FROM fact_customer_month").df())
print(wh.sql("SELECT * FROM fact_customer_month WHERE customer_key = 1 ORDER BY month LIMIT 6").df())

                     t  rows
0           fact_sales  2240
1  fact_customer_month  3540
       month  customer_key  revenue  invoices  lifetime_revenue
0 2009-01-01             1      0.0         0               0.0
1 2009-02-01             1      0.0         0               0.0
2 2009-03-01             1      0.0         0               0.0
3 2009-04-01             1      0.0         0               0.0
4 2009-05-01             1      0.0         0               0.0
5 2009-06-01             1      0.0         0               0.0


The periodic snapshot has a row for **every** customer and month, including months without purchases (revenue 0). That makes questions like "how many customers were inactive last month?" trivial, which is awkward with transaction facts alone.

An **accumulating snapshot** tracks a process with milestones. Taxi trips have two (pickup and drop-off); a delivery process would have more (ordered, packed, shipped, delivered), and each row is updated when a milestone is reached.

In [4]:
wh.execute(f"""
CREATE TABLE fact_trip_lifecycle AS
SELECT row_number() OVER () AS trip_key, tpep_pickup_datetime AS requested_at, tpep_pickup_datetime AS picked_up_at,
       tpep_dropoff_datetime AS dropped_off_at,
       CASE WHEN payment_type = 4 THEN NULL ELSE tpep_dropoff_datetime END AS paid_at,          -- disputes never complete payment
       date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS ride_minutes, total_amount
FROM read_parquet('{datasets.taxi("2024-01")}') USING SAMPLE 100000 ROWS (reservoir, 42)
""")
print(wh.sql("""SELECT count(*) AS trips, count(paid_at) AS paid, count(*) - count(paid_at) AS open_disputes,
                       round(avg(ride_minutes), 1) AS avg_ride_min FROM fact_trip_lifecycle""").df())

    trips   paid  open_disputes  avg_ride_min
0  100000  98500           1500          15.6


## 4. Slowly Changing Dimensions

Customers move, products change category, employees change teams. How should the dimension handle it?

| Type | Behaviour | History |
| :--- | :--- | :--- |
| **0** | never change (original value) | fixed |
| **1** | overwrite | lost |
| **2** | new row with validity dates and a new surrogate key | full |
| **3** | extra column with the previous value | one step |

The cell implements types 1, 2 and 3 for a customer who moves from São Paulo to Curitiba on 2024-07-01.

In [5]:
wh.execute("""
CREATE TABLE dim_customer AS
SELECT row_number() OVER (ORDER BY CustomerId) AS customer_sk, CustomerId AS customer_id, FirstName || ' ' || LastName AS name,
       City AS city, NULL::VARCHAR AS previous_city, DATE '2009-01-01' AS valid_from, DATE '9999-12-31' AS valid_to, TRUE AS is_current
FROM src.Customer
""")
cid = wh.sql("SELECT customer_id FROM dim_customer WHERE city = 'São Paulo' ORDER BY customer_id LIMIT 1").fetchone()[0]

def show(title):
    print(title)
    print(wh.sql(f"SELECT customer_sk, customer_id, city, previous_city, valid_from, valid_to, is_current FROM dim_customer "
                 f"WHERE customer_id = {cid} ORDER BY valid_from").df().to_string(index=False), "\n")

show("before:")
wh.execute("CREATE TABLE dim_customer_t1 AS SELECT * FROM dim_customer")
wh.execute(f"UPDATE dim_customer_t1 SET city = 'Curitiba' WHERE customer_id = {cid}")                     # type 1
wh.execute(f"UPDATE dim_customer SET previous_city = city, city = 'Curitiba' WHERE customer_id = {cid} AND is_current")   # type 3 columns
wh.execute(f"UPDATE dim_customer SET city = previous_city, previous_city = NULL WHERE customer_id = {cid}")               # undo for type 2 demo
wh.execute(f"""                                                                                          -- type 2: close + insert
UPDATE dim_customer SET valid_to = DATE '2024-06-30', is_current = FALSE WHERE customer_id = {cid} AND is_current;
INSERT INTO dim_customer
SELECT (SELECT max(customer_sk) + 1 FROM dim_customer), customer_id, name, 'Curitiba', city, DATE '2024-07-01', DATE '9999-12-31', TRUE
FROM dim_customer WHERE customer_id = {cid} AND valid_to = DATE '2024-06-30';
""")
show("after a type 2 change (with the old city kept as previous_city, type 3 style):")
print("type 1 table (history lost):", wh.sql(f"SELECT city FROM dim_customer_t1 WHERE customer_id = {cid}").fetchone())

before:
 customer_sk  customer_id      city previous_city valid_from   valid_to  is_current
          10           10 São Paulo          None 2009-01-01 9999-12-31        True 

after a type 2 change (with the old city kept as previous_city, type 3 style):
 customer_sk  customer_id      city previous_city valid_from   valid_to  is_current
          10           10 São Paulo           NaN 2009-01-01 2024-06-30       False
          60           10  Curitiba     São Paulo 2024-07-01 9999-12-31        True 

type 1 table (history lost): ('Curitiba',)


A fact row must join to the dimension version that was valid **at the time of the event**: `ON f.customer_id = d.customer_id AND f.date BETWEEN d.valid_from AND d.valid_to`, or, better, the fact row stores the surrogate key `customer_sk` that was current when it was loaded.

## 5. Degenerate and Junk Dimensions

- A **degenerate dimension** is an identifier stored in the fact table without its own dimension table (`invoice_number`): useful for grouping lines of one invoice.
- A **junk dimension** combines several low-cardinality flags (payment type, store-and-forward flag, rate code) into one small dimension with one row per combination, instead of many tiny dimensions.

In [6]:
junk = wh.sql(f"""
SELECT row_number() OVER (ORDER BY payment_type, store_and_fwd_flag, RatecodeID) AS trip_profile_key, *
FROM (SELECT payment_type, store_and_fwd_flag, RatecodeID, count(*) AS trips
      FROM read_parquet('{datasets.taxi("2024-01")}') GROUP BY ALL)
""").df()
print(f"junk dimension 'trip_profile': {len(junk)} combinations replace 3 separate flag columns")
junk.sort_values("trips", ascending=False).head(6)

junk dimension 'trip_profile': 49 combinations replace 3 separate flag columns


,trip_profile_key,payment_type,store_and_fwd_flag,RatecodeID,trips
1,2,1,N,1,2176294
13,14,2,N,1,416048
0,1,0,NaN,<NA>,140162
2,3,1,N,2,79037
38,39,4,N,1,42631
6,7,1,N,99,28646


*Insight: Design the warehouse process by process with a bus matrix, declare the grain of every fact table first, share conformed dimensions, and decide explicitly how each dimension attribute handles change (SCD type 0/1/2/3). Most reporting disagreements trace back to an undeclared grain or an unplanned dimension change.*

## Real-World Use Case or Analogy

A bus matrix is like a **city's public transport map**: each line (business process) stops at shared stations (conformed dimensions). Because the stations are shared, passengers (analysts) can change lines and combine journeys (compare sales with support tickets per customer).

Kimball's method, published in *The Data Warehouse Toolkit* (1996, 3rd edition 2013), is still the default for analytics warehouses; dbt projects (Lesson 03) usually implement exactly these fact and dimension patterns.